<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/14_Menor_Caminho_Dijkstra_e_Roteamento_Dinamico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 14 - Notebook: Algoritmo de Dijkstra e Roteamento Dinâmico em Tempo Real na Linha de Envase

Neste notebook implementamos o **Algoritmo de Dijkstra** utilizando fila de prioridade binária (`heapq`) para roteamento ótimo de fluidos e contingência na **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)**.


In [2]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=3.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_padrao():
    nos = [
        "TS1_Suprimento", "TS2_Auxiliar", "VS1_Succao",
        "BC1_Bomba", "BC2_Bomba", "AS1_Acumulador",
        "VS2_Envase", "SQ2_Medicao", "EST_Envase"
    ]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("TS1_Suprimento", "VS1_Succao", 5.0, "VS1", 3.0)
    g.adicionar_tubulacao("TS2_Auxiliar", "VS1_Succao", 6.0, "VS1_AUX", 3.0)
    g.adicionar_tubulacao("VS1_Succao", "BC1_Bomba", 3.0, "SP1_SQ1", 3.0)
    g.adicionar_tubulacao("VS1_Succao", "BC2_Bomba", 4.0, "SP1_B_SQ1", 3.0)
    g.adicionar_tubulacao("BC1_Bomba", "AS1_Acumulador", 8.0, "CHECK_V1", 2.5)
    g.adicionar_tubulacao("BC2_Bomba", "AS1_Acumulador", 10.0, "CHECK_V2", 2.5)
    g.adicionar_tubulacao("AS1_Acumulador", "VS2_Envase", 10.0, "VS2", 2.0)
    g.adicionar_tubulacao("VS2_Envase", "SQ2_Medicao", 2.0, "SQ2_IN", 1.5)
    g.adicionar_tubulacao("AS1_Acumulador", "SQ2_Medicao", 14.0, "XV_BYPASS", 2.0)
    g.adicionar_tubulacao("SQ2_Medicao", "EST_Envase", 1.5, "BICO_FILL", 1.5)
    return g

import heapq
from typing import Dict, List, Tuple, Optional, Set

rede = criar_rede_padrao()

class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str, destino: str,
                               bloqueios: Optional[Set[str]] = None) -> Tuple[float, List[str]]:
        if bloqueios is None: bloqueios = set()
        if origem in bloqueios or destino in bloqueios: return float('inf'), []

        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]

        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]: continue
            if u == destino: break

            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))

        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []

roteador = RoteadorDijkstra(rede)
custo_nom, rota_nom = roteador.calcular_menor_caminho("TS1_Suprimento", "EST_Envase")
print(f"Rota Nominal Ótima: {' -> '.join(rota_nom)} | Comprimento: {custo_nom:.1f} m")
assert custo_nom < 100.0


Rota Nominal Ótima: TS1_Suprimento -> VS1_Succao -> BC1_Bomba -> AS1_Acumulador -> VS2_Envase -> SQ2_Medicao -> EST_Envase | Comprimento: 29.5 m
